# Task 3.1.5: Object Tracking using Sparse Optical Flow

In [1]:
# Importing required libraries, which include:
# OpenCV library used for the many tasks in computer vision, from loading images, to processing, detecting, shapes, tracking objects etc.:
import cv2
import os
import numpy as np

In [2]:
# TASK1: Create a program which enables user to manually select exactly two keypoints from the image.
# Create a window where the user can select keypoints using the mouse and save them in the list. This will be used later as initial points for the optical flow.T
# As keypoints are used in the tracking, when selecting them, select one on top-left part of the object and one on the bottom-right part of the object.mro

# Hint: create a callback function for the mouse event 'click' that saves the selected keypoint in the global list. 
# Clicking event is cv2.EVENT_LBUTTONDOWN.

# Create an empty list to store the selected keypoints
keypoints = []

def click(event, x, y, flags, param):
    """
    A callback function for mouse event - saving selected keypoints.
    """
    global keypoints
    
    if event == cv2.EVENT_LBUTTONDOWN and len(keypoints) < 2:
        # Save the selected keypoint in the list
        keypoints.append((x, y))
        print(f"Selected keypoint: {(x, y)}")
        
    



# Load the image into a numpy array
frame = cv2.imread("./data/got10k/sequence_example/00000001.jpg")

# Create a new window and set the mouse callback function to select 2 key points.
cv2.imshow("Select keypoints", frame)
cv2.setMouseCallback("Select keypoints", click)

while len(keypoints) < 2:
    cv2.waitKey(1)

cv2.destroyAllWindows()

for _ in range(5):
    cv2.waitKey(1)

print(keypoints)

Selected keypoint: (605, 466)
Selected keypoint: (262, 269)
[(605, 466), (262, 269)]


In [3]:
# TASK2: Sparse optical flow. Use the Lucas-Kanade optical flow algorithm to track an object in a video sequence. 

# Hint: You can follow next steps: 
#        Loop through all the frames in the directory, except the first one - that is the image that we got inital bounding box points from in the previous snippet. 
#        Convert each frame to grayscale.
#        Use the points from previous frame to calculate the optical flow between the previous and current frame. 
#        If two points were successfully tracked, draw a bounding box around the object. 
#        Display the frame.

# Set the path to the directory containing the frames
frame_dir = "./data/got10k/sequence_example/"
frame_filenames = sorted(os.listdir(frame_dir))

first_frame = cv2.imread(os.path.join(frame_dir, frame_filenames[0]))
prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)

prev_points = np.array(keypoints, dtype=np.float32).reshape(-1, 1, 2)

# Perform tracking using sparse optical flow

for filename in frame_filenames[1:]:
    frame = cv2.imread(os.path.join(frame_dir, filename))
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    new_points, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_points, None)
    
    good_new = new_points[status == 1]
    
    if(len(good_new) == 2):
        x1, y1 = good_new[0]
        x2, y2 = good_new[1]
        
        
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        prev_points = good_new.reshape(-1, 1, 2)
        
    cv2.imshow("Sparse optical Tracking", frame)
    
    key = cv2.waitKey(30) & 0xFF
    
    if key == ord("q") or cv2.getWindowProperty("Sparse optical Tracking", cv2.WND_PROP_VISIBLE) < 1:
        break

    prev_gray = gray.copy()

cv2.destroyAllWindows()

for _ in range(20):
    cv2.waitKey(1)

: 